# Setup

In [1]:
!pip install -q -U "transformers==4.49.0" "accelerate>=1.3.0" "peft>=0.14" "bitsandbytes>=0.45"
!pip install -q -U "trl==0.13.0"
!pip install -q -U qwen-vl-utils bert-score rouge-score "torchao>=0.16"
!pip install -q "datasets<4.0.0"

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.0/44.0 kB 1.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.0/10.0 MB 67.1 MB/s eta 0:00:00:00:01:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 383.7/383.7 kB 20.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 680.7/680.7 kB 31.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 31.7 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 32.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.1/3.1 MB 75.8 MB/s eta 0:00:00:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 293.4/293.4 kB 5.9 MB/s eta 0:00:00a 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.1/61.1 kB 2.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 43.8 MB/s eta 0:00:0000:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 36.3/36.3 MB 48.3 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 491.5/491.5 kB 9.2 MB/s eta 

In [2]:
import os
import torch
import random
import logging
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from tqdm import tqdm

import nltk
from datasets import load_dataset, Dataset
from transformers import Qwen2_5_VLForConditionalGeneration, AutoProcessor
from peft import PeftModel
from qwen_vl_utils import process_vision_info
from huggingface_hub import login
from kaggle_secrets import UserSecretsClient

from nltk.translate.bleu_score import corpus_bleu, SmoothingFunction
from rouge_score import rouge_scorer
from bert_score import BERTScorer

2026-05-25 11:38:43.794667: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1779709124.224670      57 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1779709124.359826      57 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1779709125.463248      57 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1779709125.463287      57 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1779709125.463290      57 computation_placer.cc:177] computation placer alr

In [3]:
nltk.download("punkt", quiet=True)
nltk.download("punkt_tab", quiet=True)

True

In [4]:
device = "cuda" if torch.cuda.is_available() else "cpu"
logging.getLogger("transformers").setLevel(logging.ERROR)

In [5]:
RANDOM_SEED = 42
os.environ["PYTHONHASHSEED"] = str(RANDOM_SEED)
random.seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)
torch.manual_seed(RANDOM_SEED)
torch.cuda.manual_seed(RANDOM_SEED)
torch.backends.cudnn.deterministic = True

In [6]:
hf_token = UserSecretsClient().get_secret("HF_TOKEN")
login(token=hf_token)

## Model

In [7]:
HF_USERNAME = "sdmikhalin"
BASE_MODEL = "Qwen/Qwen2.5-VL-3B-Instruct"
LORA_ADAPTER = f"{HF_USERNAME}/qwen25vl-3b-ecommerce-lora"
DATASET_REPO = f"{HF_USERNAME}/ecommerce-10k"

MIN_PIXELS = 128 * 28 * 28
MAX_PIXELS = 256 * 28 * 28

# Dataset loading

In [8]:
print(f"Loading dataset from: {DATASET_REPO}")
ds = load_dataset(DATASET_REPO)
test_data = ds["test"]

print(f"Test samples: {len(test_data)}")
print(f"Category distribution: {pd.Series(test_data['category']).value_counts().to_dict()}")

Loading dataset from: sdmikhalin/ecommerce-10k


README.md:   0%|          | 0.00/455 [00:00<?, ?B/s]

data/train-00000-of-00001.parquet:   0%|          | 0.00/398M [00:00<?, ?B/s]

data/test-00000-of-00001.parquet:   0%|          | 0.00/20.8M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/9442 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/497 [00:00<?, ? examples/s]

Test samples: 497
Category distribution: {'clothing': 197, 'electronics': 167, 'accessories': 133}


# Model and Processor Initialization

In [9]:
print(f"Loading base model: {BASE_MODEL}")
base_model = Qwen2_5_VLForConditionalGeneration.from_pretrained(
    BASE_MODEL,
    torch_dtype=torch.bfloat16,
    device_map="auto",
    attn_implementation="sdpa",
)

print(f"Applying LoRA adapter: {LORA_ADAPTER}")
model = PeftModel.from_pretrained(base_model, LORA_ADAPTER)
model.eval()

processor = AutoProcessor.from_pretrained(
    LORA_ADAPTER,
    min_pixels=MIN_PIXELS,
    max_pixels=MAX_PIXELS,
)

print(f"VRAM usage: {torch.cuda.memory_allocated() / 1e9:.1f} GB")

Loading base model: Qwen/Qwen2.5-VL-3B-Instruct


config.json: 0.00B [00:00, ?B/s]

model.safetensors.index.json: 0.00B [00:00, ?B/s]

model-00001-of-00002.safetensors:   0%|          | 0.00/3.98G [00:00<?, ?B/s]

model-00002-of-00002.safetensors:   0%|          | 0.00/3.53G [00:00<?, ?B/s]

...

tokenizer.json:   0%|          | 0.00/11.4M [00:00<?, ?B/s]

added_tokens.json:   0%|          | 0.00/605 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/613 [00:00<?, ?B/s]

chat_template.json: 0.00B [00:00, ?B/s]

VRAM usage: 4.0 GB


# Prompt Templates  and Generation Function

In [10]:
SYSTEM_PROMPT = (
    "You are an e-commerce product description assistant for an online marketplace.\n"
    "Generate concise product titles in this format: [Brand] [Gender/Audience] [Color/Material] [Type].\n"
    "Keep it under 12 words. Be specific. Avoid filler words like 'this', 'a', 'an', 'image of'."
)

USER_PROMPTS = {
    "clothing": "Describe this clothing item as a marketplace product title. Include brand, gender, color, and type.",
    "electronics": "Describe this electronic device as a marketplace product title. Include brand, model, color, and key specifications.",
    "accessories": "Describe this accessory as a marketplace product title. Include brand, gender, color, material, and type.",
}
DEFAULT_USER_PROMPT = "Describe this product as a concise marketplace title."

def build_user_prompt(category):
    return USER_PROMPTS.get(category, DEFAULT_USER_PROMPT)

def generate_with_mode(image, category, use_lora=True, max_new_tokens=128):
    if image is None:
        return ""

    image = image.convert("RGB")
    user_prompt = build_user_prompt(category)

    messages = [
        {"role": "system", "content": [{"type": "text", "text": SYSTEM_PROMPT}]},
        {"role": "user", "content": [
            {"type": "image", "image": image},
            {"type": "text", "text": user_prompt},
        ]},
    ]

    text = processor.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    image_inputs, _ = process_vision_info(messages)
    inputs = processor(
        text=[text], images=image_inputs,
        return_tensors="pt", padding=True,
    ).to(model.device)

    gen_kwargs = dict(
        max_new_tokens=max_new_tokens,
        do_sample=False,
        num_beams=3,
        repetition_penalty=1.1,
        length_penalty=0.8,
        no_repeat_ngram_size=3,
    )

    if use_lora:
        with torch.no_grad():
            output = model.generate(**inputs, **gen_kwargs)
    else:
        with model.disable_adapter(), torch.no_grad():
            output = model.generate(**inputs, **gen_kwargs)

    generated = output[:, inputs["input_ids"].shape[1]:]
    return processor.batch_decode(generated, skip_special_tokens=True)[0].strip()

# Inference

In [11]:
results = []
for item in tqdm(test_data, desc="Running inference"):
    pred_lora = generate_with_mode(item["image"], item["category"], use_lora=True)
    pred_base = generate_with_mode(item["image"], item["category"], use_lora=False)
    
    results.append({
        "category": item["category"],
        "reference": item["text"],
        "generated_finetuned": pred_lora,
        "generated_baseline": pred_base,
    })

results_df = pd.DataFrame(results)
results_df.to_csv("comparison_results.csv", index=False)

Running inference:   0%|          | 0/497 [00:00<?, ?it/s]/usr/local/lib/python3.12/dist-packages/transformers/generation/configuration_utils.py:629: UserWarning: `do_sample` is set to `False`. However, `temperature` is set to `1e-06` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `temperature`.
  warnings.warn(
Running inference:   0%|          | 0/497 [00:08<?, ?it/s]


KeyboardInterrupt: 

# Metrics Calculation

In [ ]:
print("Initializing BERTScorer...")
bert_scorer = BERTScorer(lang="en", rescale_with_baseline=False)

def calculate_metrics(df, pred_column):
    refs = [r.lower().strip() for r in df["reference"].tolist()]
    gens = [g.lower().strip() for g in df[pred_column].tolist()]

    smoothie = SmoothingFunction().method1
    refs_tok = [[r.split()] for r in refs]
    gens_tok = [g.split() for g in gens]
    
    bleu1 = corpus_bleu(refs_tok, gens_tok, weights=(1, 0, 0, 0), smoothing_function=smoothie)
    bleu4 = corpus_bleu(refs_tok, gens_tok, weights=(0.25,) * 4, smoothing_function=smoothie)

    scorer = rouge_scorer.RougeScorer(["rougeL"], use_stemmer=True)
    rougeL = np.mean([scorer.score(r, g)["rougeL"].fmeasure for r, g in zip(refs, gens)])

    _, _, F1 = bert_scorer.score(gens, refs)

    return {
        "BLEU-1": bleu1,
        "BLEU-4": bleu4,
        "ROUGE-L": rougeL,
        "BERTScore-F1": F1.mean().item(),
    }

# --- GLOBAL METRICS ---
print("\n" + "=" * 30)
print("OVERALL METRICS")
print("=" * 30)
m_lora = calculate_metrics(results_df, "generated_finetuned")
m_base = calculate_metrics(results_df, "generated_baseline")

print(f"{'Metric':<15} {'Zero-shot':<15} {'Fine-tuned':<15} {'Improvement':<15}")
for m in ["BLEU-1", "BLEU-4", "ROUGE-L", "BERTScore-F1"]:
    b, f = m_base[m], m_lora[m]
    imp = f"+{100 * (f - b) / b:.1f}%" if b > 0 else "N/A"
    print(f"{m:<15} {b:<15.4f} {f:<15.4f} {imp:<15}")

# --- CATEGORICAL METRICS ---
print("\n" + "=" * 30)
print("METRICS BY CATEGORY")
print("=" * 30)
for cat in sorted(results_df["category"].unique()):
    cat_df = results_df[results_df["category"] == cat]
    print(f"\nCategory: {cat.upper()} (n={len(cat_df)})")
    cm_base = calculate_metrics(cat_df, "generated_baseline")
    cm_lora = calculate_metrics(cat_df, "generated_finetuned")
    for m in ["BLEU-1", "BLEU-4", "ROUGE-L", "BERTScore-F1"]:
        b, f = cm_base[m], cm_lora[m]
        imp = f"+{100 * (f - b) / b:.1f}%" if b > 0 else "N/A"
        print(f"{m:<15} {b:<15.4f} {f:<15.4f} {imp:<15}")

# Visualization and Examples Exports

In [ ]:
EXAMPLES_DIR = "comparison_examples"
os.makedirs(EXAMPLES_DIR, exist_ok=True)

N_PER_CATEGORY = 3
samples_to_show = []

for cat in sorted(set(test_data["category"])):
    cat_indices = [i for i in range(len(test_data)) if test_data[i]["category"] == cat]
    chosen = random.sample(cat_indices, min(N_PER_CATEGORY, len(cat_indices)))
    samples_to_show.extend([(i, test_data[i]) for i in chosen])

n_rows = len(samples_to_show)
fig, axes = plt.subplots(n_rows, 1, figsize=(12, 5.5 * n_rows))
if n_rows == 1: axes = [axes]

for ax, (idx, item) in zip(axes, samples_to_show):
    pred_lora = results_df.iloc[idx]["generated_finetuned"]
    pred_base = results_df.iloc[idx]["generated_baseline"]

    ax.imshow(item["image"])
    ax.axis("off")
    title = (
        f"Category: {item['category']}\n"
        f"{'-'*40}\n"
        f"Reference:  {item['text']}\n"
        f"Fine-tuned: {pred_lora}\n"
        f"Zero-shot:  {pred_base}"
    )
    ax.set_title(title, fontsize=10, loc="left", pad=15, family="monospace")

plt.tight_layout()
plt.savefig(f"{EXAMPLES_DIR}/comparison_overview.png", dpi=80, bbox_inches="tight")
plt.show()